In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
google_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

In [ ]:
print("GROQ_API_KEY configured:", bool(groq_api_key))
print("GOOGLE_API_KEY configured:", bool(google_api_key))


In [9]:
import os
from sarvamai import SarvamAI

client = SarvamAI(api_subscription_key=os.environ["SARVAM_API_KEY"])
model1 = os.getenv('SARVAM_MODEL')

response = client.chat.completions_v2(
    model='glm5.3',  # or "gemma4", "deepseekv4.1-flash-1017", "kimi-k3.1-pro", ...
    messages=[
        {"role": "user", "content": "Answer , 2+2."}
    ],
    max_tokens=1000,
)

print(response.choices[0].message.content)


4


In [16]:
response.choices[0].message.reasoning_content

"The user is asking me to answer 2+2. This is a simple arithmetic question.\n\n2+2 = 4.\n\nThe format should be extremely brief — just the answer, possibly with minimal context. There's no need for lengthy explanation for such a trivial question. A one-word or very short answer is appropriate."

In [11]:
dict(response)

{'id': '20261108_c8557ad2-3fd4-4eb8-8fec-84d95f38a432',
 'choices': [Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionResponseMessage(content='4', refusal=None, reasoning_content="The user is asking me to answer 2+2. This is a simple arithmetic question.\n\n2+2 = 4.\n\nThe format should be extremely brief — just the answer, possibly with minimal context. There's no need for lengthy explanation for such a trivial question. A one-word or very short answer is appropriate.", role='assistant', tool_calls=None, annotations=None, audio=None, function_call=None))],
 'created': 1794115041,
 'model': 'glm5.3',
 'object': 'chat.completion',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': CompletionUsage(completion_tokens=68, prompt_tokens=19, total_tokens=87, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=65, accepted_prediction_tokens=None, audio_tokens=None, rejected_prediction_tokens=None), prompt_tokens_details=CompletionUsagePromptT

In [15]:
dict(response.choices[0])

{'finish_reason': 'stop',
 'index': 0,
 'logprobs': None,
 'message': ChatCompletionResponseMessage(content='4', refusal=None, reasoning_content="The user is asking me to answer 2+2. This is a simple arithmetic question.\n\n2+2 = 4.\n\nThe format should be extremely brief — just the answer, possibly with minimal context. There's no need for lengthy explanation for such a trivial question. A one-word or very short answer is appropriate.", role='assistant', tool_calls=None, annotations=None, audio=None, function_call=None)}

In [17]:
dict(response.choices[0].message)

{'content': '4',
 'refusal': None,
 'reasoning_content': "The user is asking me to answer 2+2. This is a simple arithmetic question.\n\n2+2 = 4.\n\nThe format should be extremely brief — just the answer, possibly with minimal context. There's no need for lengthy explanation for such a trivial question. A one-word or very short answer is appropriate.",
 'role': 'assistant',
 'tool_calls': None,
 'annotations': None,
 'audio': None,
 'function_call': None}

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

## Tool for pereforming data analysis on the database

In [ ]:
import os
import sys
from io import StringIO
import pandas as pd
from sqlalchemy import create_engine

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent, AgentExecutor

# ---------------------------------------------------------------------------
# 1. Database Setup & Tool Definitions
# ---------------------------------------------------------------------------
DB_URL = "sqlite:///payment_data.db"
engine = create_engine(DB_URL)
TABLE_NAME = "merchant_settlement_reports_for_Nov-06_to_Nov-11"

@tool
def execute_data_analysis(python_code: str) -> str:
    """
    Executes Python/Pandas code to analyze merchant payment settlement data.
    
    Environment variables available in script:
      - `pd`: Pandas library
      - `engine`: SQLAlchemy engine connected to SQLite
      - `table_name`: Name of the settlement table
      
    Instructions:
      - Read data using: df = pd.read_sql(f'SELECT * FROM "{table_name}"', engine)
      - Always use print() to output results, numbers, aggregates, and findings.
    """
    # Create execution namespace
    local_scope = {
        "pd": pd,
        "engine": engine,
        "table_name": TABLE_NAME,
    }

    # Redirect stdout to capture print() statements
    stdout_buffer = StringIO()  
    original_stdout = sys.stdout

    try:
        sys.stdout = stdout_buffer
        exec(python_code, {}, local_scope)
        output = stdout_buffer.getvalue()
        return output.strip() if output.strip() else "Code executed successfully with no printed output."
    except Exception as err:
        return f"Execution error: {str(err)}"
    finally:
        sys.stdout = original_stdout


# ---------------------------------------------------------------------------
# 2. Initialize Gemini Model & Agent
# ---------------------------------------------------------------------------
tools = [execute_data_analysis]

# Initialize Gemini model with function-calling support
llm = ChatGroq(
    model="openai/gpt-oss-120b", # or "gemini-1.5-flash"
    temperature=0.0,
    api_key=os.getenv("GROQ_API_KEY")
)

# Define the conversational prompt template
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert financial and merchant data analyst.\n"
        "You have access to a SQLite database via the `execute_data_analysis` tool.\n\n"
        "Rules:\n"
        "1. Never guess or approximate numbers. Always write Python/Pandas code using the tool to calculate metrics.\n"
        "2. Compute descriptive statistics, totals, averages, volume by instrument, and identify any anomalies or outliers.\n"
        "3. Provide direct, concise, and business-focused insights based strictly on the computation results."
    ),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Build the tool-calling agent and executor loop
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


# ---------------------------------------------------------------------------
# 3. Running Analysis Queries
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    query = (
        "Analyze payment instruments by total volume and transaction count. "
        "Also check if there are any unusual transactions or fee deductions."
    )
    
    response = agent_executor.invoke({"input": query})
    print("\n--- FINAL INSIGHT ---")
    print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `execute_data_analysis` with `{'python_code': 'import pandas as pd\ndf = pd.read_sql(f\'SELECT * FROM "{table_name}"\', engine)\nprint(\'Columns:\', df.columns.tolist())\nprint(\'First rows:\', df.head())\nprint(\'Row count:\', len(df))'}`


Columns: ['paymenttype', 'merchantreferenceid', 'phonepereferenceid', 'from', 'instrument', 'creationdate', 'transactiondate', 'settlementdate', 'bankreferenceno', 'amount', 'fee', 'igst', 'cgst', 'sgst']
First rows:   paymenttype                  merchantreferenceid       phonepereferenceid  \
0     PAYMENT           AC232611052048537404856412  T2611052048537451632553   
1     PAYMENT  YBN20261105095145695682301952080000  T2611052038506124694607   
2     PAYMENT           AC222611052001549505012423  T2611052001549631671442   
3     PAYMENT  HDFf71a682b4d13469a86215051dbe9e2ee  T2611051953265948882983   
4     PAYMENT           AC222611051942456077161368  T2611051942456208497406   

               

In [11]:
response = agent_executor.invoke({"input": "When did i got more collections.and also based on this trends predict how much do i get for the next day."})



> Entering new AgentExecutor chain...

Invoking: `execute_data_analysis` with `{'python_code': '\nimport pandas as pd\n# Load data\ndf = pd.read_sql(\'SELECT * FROM "settlements"\', engine)\n# Assume there is a column for collection amount, e.g., \'collection_amount\' and a date column \'settlement_date\'\n# If column names differ, list columns\nprint(\'Columns:\', df.columns.tolist())\n'}`


Execution error: (sqlite3.OperationalError) no such table: settlements
[SQL: SELECT * FROM "settlements"]
(Background on this error at: https://sqlalche.me/e/21/e3q8)
Invoking: `execute_data_analysis` with `{'python_code': '\nimport pandas as pd\n# List tables in SQLite\ntables = pd.read_sql("SELECT name FROM sqlite_master WHERE type=\'table\'", engine)\nprint(tables)\n'}`


name
0  merchant_settlement_reports_for_Nov-06_to_Nov-11
Invoking: `execute_data_analysis` with `{'python_code': '\nimport pandas as pd\ntable_name = \'merchant_settlement_reports_for_Nov-06_to_Nov-11\'\ndf = pd.read_sql(f\'

In [13]:
response['output']

'**Highest collection day**\n\n| Settlement Date | Total\u202fAmount Collected |\n|-----------------|-----------------------|\n| **09‑Nov‑2026** | **13,330** |\n\nYour peak collection within the available window (06‑Nov‑2026\u202fto\u202f11‑Nov‑2026) occurred on **09\u202fNov\u202f2026**, where the daily total was **13,330** units (currency not specified).\n\n---\n\n**Trend‑based forecast for the next day (12‑Nov‑2026)**  \n\nUsing a simple linear trend (ordinary least‑squares fit on the daily totals), the model predicts:\n\n- **Predicted total amount on 12‑Nov‑2026:** **7,362.47**\n\n*Interpretation*: After the spike on 09‑Nov‑2026, daily collections have been trending downward. The linear model therefore forecasts a moderate drop to roughly **7.36\u202fk** for the following day.\n\n---\n\n**Key take‑aways**\n\n1. **Peak performance** was on 09‑Nov‑2026 (13.3\u202fk).  \n2. **Recent daily totals** (06‑Nov to 11‑Nov) show a decreasing pattern after the peak: 5,065 → 4,695 → 4,187 → **1